# ARC-AGI Solver — Colab Training

Trains the decoder-only transformer on a chosen category of ARC tasks using RE-ARC synthetic data.

**Default target:** `STRUCTURE_UNCHANGED` — 52 tasks where the zero/non-zero mask is identical in input and output (colour-only changes). This is the simplest category and a good end-to-end proof of concept.

**Runtime:** Use a GPU runtime — Runtime → Change runtime type → T4 GPU (free) or A100 (Pro).

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be very slow on CPU.')

In [ ]:
# ── Cell 2: Clone repo ──────────────────────────────────────────────────────
import os
REPO = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
if not os.path.isdir(REPO):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git')
else:
    os.system(f'git -C {REPO} pull --ff-only')
os.chdir(REPO)
print(f'Working directory: {os.getcwd()}')
os.system('git log --oneline -3')

In [ ]:
# ── Cell 3: Download ARC training data ──────────────────────────────────────
import io, urllib.request, zipfile
from pathlib import Path

DATA_DIR = Path('data')
if (DATA_DIR / 'training').exists() and len(list((DATA_DIR / 'training').glob('*.json'))) >= 400:
    print(f"ARC training data already present ({len(list((DATA_DIR/'training').glob('*.json')))} tasks).")
else:
    ARC_URL = 'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    print(f'Downloading ARC-AGI dataset...')
    with urllib.request.urlopen(ARC_URL) as r:
        raw = r.read()
    print(f'Downloaded {len(raw)/1e6:.1f} MB')
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for split in ('training', 'evaluation'):
            dest = DATA_DIR / split
            dest.mkdir(exist_ok=True)
            members = [n for n in zf.namelist() if f'data/{split}/' in n and n.endswith('.json')]
            for m in members:
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list((DATA_DIR/'training').glob('*.json')))} tasks")
    print(f"evaluation: {len(list((DATA_DIR/'evaluation').glob('*.json')))} tasks")

In [ ]:
# ── Cell 4: Download RE-ARC synthetic data ───────────────────────────────────
# RE-ARC provides 1,000 synthetic examples per task (used for training).
# Downloads ~110 MB zip and extracts all 400 task files.
from pathlib import Path
RE_ARC_DIR = Path('data/re_arc')
if RE_ARC_DIR.exists() and len(list(RE_ARC_DIR.glob('*.json'))) >= 400:
    print(f"RE-ARC already present ({len(list(RE_ARC_DIR.glob('*.json')))} files).")
else:
    import subprocess
    subprocess.run(['python', 'scripts/download_re_arc.py'], check=True)

In [ ]:
# ── Cell 5: Generate category labels ────────────────────────────────────────
import json, subprocess
from pathlib import Path
if not Path('results/categories_training.json').exists():
    print('Generating category labels...')
    subprocess.run(['python', '-m', 'src.explore'], check=True)
else:
    print('categories_training.json already present.')
data = json.loads(Path('results/categories_training.json').read_text())
su_tasks = [tid for tid, cats in data.items() if 'STRUCTURE_UNCHANGED' in cats]
print(f'STRUCTURE_UNCHANGED tasks: {len(su_tasks)}')

In [ ]:
# ── Cell 5b: Sanity check — EXTEND_RECT tasks ───────────────────────────────
import json
from pathlib import Path

cats = json.loads(Path('results/categories_training.json').read_text())
er_tasks = [tid for tid, task_cats in cats.items() if 'EXTEND_RECT' in task_cats]
print(f'EXTEND_RECT task count: {len(er_tasks)}')
print(f'Sample task IDs: {er_tasks[:5]}')

re_arc_dir = Path('data/re_arc')
missing = [tid for tid in er_tasks if not (re_arc_dir / f'{tid}.json').exists()]
if missing:
    print(f'WARNING: {len(missing)} tasks missing RE-ARC data: {missing[:10]}')
else:
    print(f'All {len(er_tasks)} EXTEND_RECT tasks have RE-ARC data in data/re_arc/')

In [ ]:
# ── Cell 6: Training configuration ──────────────────────────────────────────
# Choose ONE of the two modes below by setting MODE.
#
# MODE = 'CATEGORY'  — train on a programmatic category (reads categories_training.json)
#                      e.g. EXTEND_RECT (66 tasks), STRUCTURE_UNCHANGED (52 tasks)
#
# MODE = 'TASK_IDS'  — train on an explicit list of task IDs (from scene-description
#                      clusters).  Set TASK_IDS and RUN_NAME below.

MODE = 'TASK_IDS'   # ← switch between 'CATEGORY' and 'TASK_IDS'

# ── CATEGORY mode ────────────────────────────────────────────────────────────
CATEGORY = 'EXTEND_RECT'

# ── TASK_IDS mode — C8 (compartment fill, 11 tasks) ─────────────────────────
RUN_NAME  = 'C8_compartment_fill'
TASK_IDS  = [
    '09629e4f', '1190e5a7', '1bfc4729', '1e32b0e9', '272f95fa',
    '29623171', '54d9e175', '6773b310', '6d0160f0', '7b6016b9', '941d9a10',
]

# ── Shared hyperparameters ────────────────────────────────────────────────────
EPOCHS          = 200
STEPS_PER_EPOCH = 200
K_CONTEXT       = 3
D_MODEL         = 256
N_LAYERS        = 6
LR              = 3e-4
WARMUP_EPOCHS   = 5
MAX_TOKENS      = 96000
SAVE_EVERY      = 25
LOG_EVERY       = 5

# ── Resolve target_tasks and run metadata ─────────────────────────────────────
import json
from pathlib import Path

if MODE == 'TASK_IDS':
    target_tasks = TASK_IDS
    run_tag      = RUN_NAME
else:
    cats_data    = json.loads(Path('results/categories_training.json').read_text())
    target_tasks = [tid for tid, cats in cats_data.items() if CATEGORY in cats]
    run_tag      = CATEGORY

LOG_FILE = f'results/train_{run_tag.lower()}.log'

print(f'Mode:      {MODE}')
print(f'Run name:  {run_tag}')
print(f'Tasks:     {len(target_tasks)}  →  {target_tasks[:5]}{"..." if len(target_tasks)>5 else ""}')
print(f'Epochs:    {EPOCHS} × {STEPS_PER_EPOCH} steps = {EPOCHS*STEPS_PER_EPOCH:,} total')
print(f'd_model:   {D_MODEL},  layers: {N_LAYERS},  k_context: {K_CONTEXT}')
print(f'LR:        {LR},  warmup: {WARMUP_EPOCHS} epochs,  max_tokens: {MAX_TOKENS}')

In [ ]:
# ── Cell 6b: Pre-tokenise training tasks ────────────────────────────────────
# Encodes every RE-ARC example once to numpy arrays on disk (data/tokenized/).
# Replaces the slow Python tokenisation loop during training with fast numpy ops.
# Typical time: ~30 s for 66 EXTEND_RECT tasks on CPU.
import subprocess, sys
from pathlib import Path

tokenized_dir = Path('data/tokenized')
already = {p.stem for p in tokenized_dir.glob('*.npz')} if tokenized_dir.exists() else set()
missing = [tid for tid in target_tasks if tid not in already]

if not missing:
    print(f'All {len(target_tasks)} tasks already pre-tokenised ({len(already)} total on disk).')
else:
    print(f'Pre-tokenising {len(missing)} tasks (already have {len(already)})...')
    cmd_tok = [sys.executable, 'scripts/pretokenize.py', '--tasks'] + missing
    result = subprocess.run(cmd_tok, capture_output=True, text=True)
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    else:
        n = len(list(tokenized_dir.glob('*.npz')))
        print(f'Done. {n} tasks tokenised in {tokenized_dir}.')

In [ ]:
# ── Cell 7: Run training ─────────────────────────────────────────────────────
# Checkpoints: checkpoints/transformer_{run_tag}_best.pt
#              checkpoints/transformer_{run_tag}_final.pt
import subprocess, sys

base_cmd = [
    sys.executable, '-u', 'scripts/train_transformer.py',
    '--epochs',          str(EPOCHS),
    '--steps-per-epoch', str(STEPS_PER_EPOCH),
    '--k-context',       str(K_CONTEXT),
    '--d-model',         str(D_MODEL),
    '--n-layers',        str(N_LAYERS),
    '--lr',              str(LR),
    '--warmup-epochs',   str(WARMUP_EPOCHS),
    '--max-tokens',      str(MAX_TOKENS),
    '--max-seq-len',     '6000',
    '--save-every',      str(SAVE_EVERY),
    '--log-every',       str(LOG_EVERY),
    '--log',             LOG_FILE,
    '--run-name',        run_tag,
]

if MODE == 'TASK_IDS':
    cmd = base_cmd + ['--task-ids'] + target_tasks
else:
    cmd = base_cmd + ['--category', CATEGORY]

print('Command:', ' '.join(cmd[:10]), '...')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 8: View training log ────────────────────────────────────────────────
from pathlib import Path
log = Path(LOG_FILE)
if log.exists():
    lines = log.read_text().splitlines()
    print(f'Log: {log} ({len(lines)} lines)')
    print('\n'.join(lines[-40:]))
else:
    print('Log file not found yet.')

In [ ]:
# ── Cell 9: Download best checkpoint ────────────────────────────────────────
from pathlib import Path
from google.colab import files

ckpt = Path(f'checkpoints/transformer_{CATEGORY}_best.pt')
if ckpt.exists():
    print(f'Downloading {ckpt} ({ckpt.stat().st_size/1e6:.1f} MB)')
    files.download(str(ckpt))
else:
    print(f'Checkpoint not found: {ckpt}')
    for p in sorted(Path('checkpoints').glob('*.pt')):
        print(f'  {p}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 10 (optional): Resume from checkpoint ───────────────────────────────
# If the Colab session disconnected mid-training, re-run cells 2-5 to
# restore the environment, then run this cell to resume.

RESUME_FROM = f'checkpoints/transformer_{CATEGORY}_best.pt'
cmd_resume = cmd + ['--resume', RESUME_FROM]
print('Resume command:', ' '.join(cmd_resume))

# Uncomment to run:
# proc = subprocess.Popen(cmd_resume, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
#                         text=True, bufsize=1)
# for line in proc.stdout:
#     print(line, end='')
# proc.wait()